# F-Scores, Harmonic Means & Balanced Classification Metrics Lab

When Precision and Recall conflict, we need a unified metric to rank models. This lab demonstrates why the **Harmonic Mean ($F_1$)** severely penalizes models that sacrifice one metric completely, explores custom weighting via $F_\beta$ ($F_{0.5}$ and $F_2$), and benchmarks **Balanced Accuracy** and the **Matthews Correlation Coefficient (MCC)** on severely imbalanced datasets.

In [ ]:
import numpy as np
from sklearn.metrics import (
    f1_score, fbeta_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix, accuracy_score
)

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Harmonic Mean ($F_1$) vs. Arithmetic Mean

Compare the arithmetic mean $\frac{P+R}{2}$ with the harmonic mean $2 \frac{P \cdot R}{P + R}$ when a model achieves high precision but terrible recall.

In [ ]:
cases = [
    (0.90, 0.10, "Extreme Divergence (P=0.90, R=0.10)"),
    (0.50, 0.50, "Equal Moderate (P=0.50, R=0.50)"),
    (0.85, 0.80, "High Balanced (P=0.85, R=0.80)")
]

print(f"{'Scenario':<38} {'Arithmetic':<12} {'Harmonic (F1)':<15} {'Penalty Delta'}")
print("-" * 78)
for p, r, label in cases:
    arith = (p + r) / 2
    harm = 2 * (p * r) / (p + r)
    delta = arith - harm
    print(f"{label:<38} {arith:<12.1%} {harm:<15.1%} -{delta:<10.1%}")

## 2. Weighting Priorities with $F_\beta$

Evaluate a medical screening model ($P=0.62, R=0.87$) across $F_{0.5}$ (precision focus), $F_1$ (balanced), and $F_2$ (recall focus).

In [ ]:
# Simulated dataset: 60 healthy, 40 sick
y_true = np.concatenate([np.zeros(60, dtype=int), np.ones(40, dtype=int)])
y_pred = np.concatenate([
    np.random.choice([0, 1], size=60, p=[0.85, 0.15]),  # False alarms
    np.random.choice([0, 1], size=40, p=[0.10, 0.90])   # High recall
])

f_half = fbeta_score(y_true, y_pred, beta=0.5)
f1 = f1_score(y_true, y_pred)
f2 = fbeta_score(y_true, y_pred, beta=2.0)

print(f"F0.5 Score (Precision priority): {f_half:.3f}")
print(f"F1 Score   (Balanced):           {f1:.3f}")
print(f"F2 Score   (Recall priority):    {f2:.3f}")
print("\nNotice: Because Recall > Precision, F2 rewards the high recall while F0.5 penalizes low precision.")

## 3. Benchmarking Imbalanced Metrics: Accuracy vs. F1 vs. MCC

Compare metric behavior on a 95:5 imbalanced dataset between a trivial dummy model (always negative) and a genuine trained model.

In [ ]:
y_imb = np.concatenate([np.zeros(95, dtype=int), np.ones(5, dtype=int)])
y_trivial = np.zeros(100, dtype=int)

y_real = np.concatenate([
    np.random.choice([0, 1], size=95, p=[0.96, 0.04]),  # 4 FP
    np.array([1, 1, 1, 1, 0])                           # 4 TP, 1 FN
])

def eval_cohort(y_t, y_p, name):
    acc = accuracy_score(y_t, y_p)
    b_acc = balanced_accuracy_score(y_t, y_p)
    f1_val = f1_score(y_t, y_p, zero_division=0)
    mcc_val = matthews_corrcoef(y_t, y_p)
    print(f"{name:<24} Acc: {acc:<7.1%} BalAcc: {b_acc:<7.1%} F1: {f1_val:<7.1%} MCC: {mcc_val:<7.3f}")

print(f"{'Model':<24} {'Accuracy':<12} {'Balanced Acc':<15} {'F1':<10} {'MCC'}")
print("-" * 70)
eval_cohort(y_imb, y_trivial, "Trivial Dummy (Zero-Rule)")
eval_cohort(y_imb, y_real, "Trained Classifier")